# TTS Integration

Text-to-speech synthesis using **Chatterbox TTS** via the GPU TTS server (port 8020).
This notebook covers baseline vs aligned dubbing modes and voice cloning from reference audio.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

# Load .env (LOGFIRE_TOKEN, HF_TOKEN, etc.)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from foreign_whispers.client import FWClient, BASELINE, ALIGNED

fw = FWClient("http://localhost:8080")
fw.healthz()

# Optional: Logfire tracing (no-op shim if unavailable)
try:
    import logfire
    logfire.configure(service_name="foreign-whispers-tts")
    print("Logfire tracing enabled.")
except Exception:
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass
    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass
    logfire = _noop()
    print("Logfire not configured — using no-op shim.")

Project root: /root/foreign-whispers


Logfire not configured — using no-op shim.


## Baseline TTS (No Alignment)

In **baseline** mode the TTS audio is concatenated segment by segment with no
duration matching against the original speech. This is the fastest mode but
segments will gradually drift out of sync with the video because each
synthesised segment may be shorter or longer than the source.

## Aligned TTS

In **aligned** mode each synthesised segment is time-stretched via
`pyrubberband` so that its duration matches the original source-language
segment window. This is slower but keeps the dubbed audio synchronised
with the video.

## Baseline vs Aligned TTS

Baseline mode synthesizes translated segments without enforcing the original timing windows. Aligned mode post-processes each segment so the final audio follows the source timeline more closely.

For the final demo, I do not re-run full TTS generation inside this notebook. The Chatterbox audio was generated on the HPC GPU and copied back into the project. This notebook verifies the generated WAV/MP4 artifacts instead of overwriting them.


## Compare Audio Durations

Load both WAV files and compare their total duration to see the effect of
time-stretching alignment.

In [2]:
# Verify final TTS WAV and final dubbed video artifacts.
# This notebook does not re-run full TTS synthesis, because the final
# Chatterbox audio was generated on the HPC GPU and copied back.

import wave
import shutil
import subprocess
from pathlib import Path

TITLE = "Strait of Hormuz disruption threatens to shake global economy"
CONFIG = "c-e5604bc"

wav_path = (
    PROJECT_ROOT
    / "pipeline_data"
    / "api"
    / "tts_audio"
    / "chatterbox"
    / CONFIG
    / f"{TITLE}.wav"
)

mp4_path = (
    PROJECT_ROOT
    / "pipeline_data"
    / "api"
    / "dubbed_videos"
    / CONFIG
    / f"{TITLE}.mp4"
)

print("Final TTS WAV:", wav_path)
print("exists:", wav_path.exists())

if wav_path.exists():
    with wave.open(str(wav_path), "rb") as wav:
        duration = wav.getnframes() / wav.getframerate()
        print(f"duration_s: {duration:.2f}")
        print(f"duration_min: {duration/60:.2f}")
        print("channels:", wav.getnchannels())
        print("sample_rate:", wav.getframerate())

print("\nFinal dubbed MP4:", mp4_path)
print("exists:", mp4_path.exists())

ffprobe = shutil.which("ffprobe")
if mp4_path.exists() and ffprobe:
    result = subprocess.run(
        [ffprobe, "-hide_banner", "-i", str(mp4_path)],
        capture_output=True,
        text=True,
    )
    print(result.stderr[:2000])
elif mp4_path.exists():
    print("ffprobe is not available in this notebook kernel.")
    print("MP4 container/audio was verified from the API container:")
    print("duration: 00:06:50.78")
    print("audio: AAC, 48000 Hz, stereo, ~190 kb/s")

Final TTS WAV: /root/foreign-whispers/pipeline_data/api/tts_audio/chatterbox/c-e5604bc/Strait of Hormuz disruption threatens to shake global economy.wav
exists: True
duration_s: 405.79
duration_min: 6.76
channels: 1
sample_rate: 24000

Final dubbed MP4: /root/foreign-whispers/pipeline_data/api/dubbed_videos/c-e5604bc/Strait of Hormuz disruption threatens to shake global economy.mp4
exists: True
ffprobe is not available in this notebook kernel.
MP4 container/audio was verified from the API container:
duration: 00:06:50.78
audio: AAC, 48000 Hz, stereo, ~190 kb/s


## Speaker Reference Voices

Chatterbox supports voice cloning from short reference WAV clips. Reference
voices are stored under `pipeline_data/speakers/` organised by language.

In [3]:
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"

if speakers_dir.exists():
    for lang_dir in sorted(speakers_dir.iterdir()):
        if lang_dir.is_dir():
            wavs = sorted(lang_dir.glob("*.wav"))
            print(f"{lang_dir.name}/  ({len(wavs)} WAV files)")
            for w in wavs:
                size_kb = w.stat().st_size / 1024
                print(f"  {w.name}  ({size_kb:.1f} KB)")
else:
    print(f"Speakers directory not found: {speakers_dir}")

es/  (3 WAV files)
  voice_0.wav  (93.8 KB)
  voice_1.wav  (93.8 KB)
  voice_2.wav  (93.8 KB)
es_alysa/  (4 WAV files)
  voice_0.wav  (93.8 KB)
  voice_1.wav  (93.8 KB)
  voice_2.wav  (93.8 KB)
  voice_3.wav  (93.8 KB)
es_before_manual_fix/  (3 WAV files)
  voice_0.wav  (93.8 KB)
  voice_1.wav  (93.8 KB)
  voice_2.wav  (93.8 KB)
es_candidates/  (25 WAV files)
  SPEAKER_00_cand0_start2.32.wav  (93.8 KB)
  SPEAKER_00_cand1_start26.64.wav  (82.5 KB)
  SPEAKER_00_cand2_start29.28.wav  (67.6 KB)
  SPEAKER_00_cand3_start31.44.wav  (75.1 KB)
  SPEAKER_00_cand4_start33.84.wav  (65.1 KB)
  SPEAKER_00_cand5_start37.84.wav  (75.0 KB)
  SPEAKER_00_cand6_start40.24.wav  (67.6 KB)
  SPEAKER_00_cand7_start42.40.wav  (85.1 KB)
  SPEAKER_01_cand0_start8.00.wav  (70.1 KB)
  SPEAKER_01_cand1_start10.24.wav  (67.6 KB)
  SPEAKER_01_cand2_start12.40.wav  (93.8 KB)
  SPEAKER_01_cand3_start15.76.wav  (77.6 KB)
  SPEAKER_01_cand4_start18.24.wav  (75.1 KB)
  SPEAKER_01_cand5_start20.64.wav  (67.6 KB)
  SPEAKER_

---

## Voice Cloning Integration

The Chatterbox client already supports reference-voice cloning through `speaker_wav`: when a reference WAV is provided, it calls `/v1/audio/speech/upload`.

In this integration, the API/service layer passes speaker-specific voice information down to the TTS engine. When diarized translated segments contain `speaker` labels, the router builds a speaker-to-voice mapping and the TTS engine uses the mapped reference WAV for each segment.

### Task 1: Understand the Existing Chatterbox Client

Read the code that already handles speaker_wav to understand what's already wired.

In [4]:
# Task 1: Read the actual Chatterbox client implementation

from pathlib import Path

tts_engine_path = PROJECT_ROOT / "api" / "src" / "services" / "tts_engine.py"
text = tts_engine_path.read_text().splitlines()

print(f"Reading: {tts_engine_path}")

keywords = [
    "CHATTERBOX_API_URL",
    "class ChatterboxClient",
    "def tts_to_file",
    "def _synthesize_default",
    "def _synthesize_with_voice",
    "def _synthesize_raw",
    "def text_file_to_speech",
]

for keyword in keywords:
    print("\n" + "=" * 80)
    print(keyword)
    print("=" * 80)

    for i, line in enumerate(text, start=1):
        if keyword in line:
            start = max(1, i - 3)
            end = min(len(text), i + 35)
            for j in range(start, end + 1):
                print(f"{j:4d}: {text[j-1]}")
            break

Reading: /root/foreign-whispers/api/src/services/tts_engine.py

CHATTERBOX_API_URL
  13: from pydub import AudioSegment
  14: 
  15: # ── Chatterbox API configuration ─────────────────────────────────────
  16: CHATTERBOX_API_URL = os.getenv("CHATTERBOX_API_URL", "http://localhost:8020")
  17: # Path to the default speaker reference WAV, relative to pipeline_data/speakers/
  18: CHATTERBOX_SPEAKER_WAV = os.getenv("CHATTERBOX_SPEAKER_WAV", "")
  19: 
  20: # Set FW_ALIGNMENT=off to use the pre-alignment baseline (legacy unclamped stretch).
  21: # Default is "on" (new clamped path). Useful for A/B comparisons.
  22: _ALIGNMENT_ENABLED = os.getenv("FW_ALIGNMENT", "on").lower() != "off"
  23: 
  24: SPEED_MIN = 0.75
  25: SPEED_MAX = 1.25
  26: # When TTS audio is less than this fraction of the target window, skip
  27: # time-stretching entirely — play at natural speed and pad with silence.
  28: # Prevents comically slow speech in windows with long narrator pauses.
  29: _STRETCH_SKIP_R

Task 1 observation:

The actual Chatterbox client lives in `api/src/services/tts_engine.py`. `ChatterboxClient.tts_to_file()` accepts `speaker_wav` via kwargs. When `speaker_wav` is provided, it calls `/v1/audio/speech/upload` and uploads the reference WAV file. This means the lower-level TTS client already supports voice cloning; the integration work is to pass speaker choices from the API/router layer down to this client.

In [5]:
# Explore what reference voices are available
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"

print("=== Available Reference Voices ===")
print(f"Global default: {(speakers_dir / 'default.wav').exists()}")
print()
for lang_dir in sorted(speakers_dir.iterdir()):
    if lang_dir.is_dir():
        wavs = sorted(lang_dir.glob("*.wav"))
        if wavs:
            print(f"{lang_dir.name}/")
            for w in wavs:
                size_kb = w.stat().st_size / 1024
                print(f"  {w.name}  ({size_kb:.1f} KB)")
        else:
            print(f"{lang_dir.name}/  (empty — needs reference WAVs)")

=== Available Reference Voices ===
Global default: False

es/
  voice_0.wav  (93.8 KB)
  voice_1.wav  (93.8 KB)
  voice_2.wav  (93.8 KB)
es_alysa/
  voice_0.wav  (93.8 KB)
  voice_1.wav  (93.8 KB)
  voice_2.wav  (93.8 KB)
  voice_3.wav  (93.8 KB)
es_before_manual_fix/
  voice_0.wav  (93.8 KB)
  voice_1.wav  (93.8 KB)
  voice_2.wav  (93.8 KB)
es_candidates/
  SPEAKER_00_cand0_start2.32.wav  (93.8 KB)
  SPEAKER_00_cand1_start26.64.wav  (82.5 KB)
  SPEAKER_00_cand2_start29.28.wav  (67.6 KB)
  SPEAKER_00_cand3_start31.44.wav  (75.1 KB)
  SPEAKER_00_cand4_start33.84.wav  (65.1 KB)
  SPEAKER_00_cand5_start37.84.wav  (75.0 KB)
  SPEAKER_00_cand6_start40.24.wav  (67.6 KB)
  SPEAKER_00_cand7_start42.40.wav  (85.1 KB)
  SPEAKER_01_cand0_start8.00.wav  (70.1 KB)
  SPEAKER_01_cand1_start10.24.wav  (67.6 KB)
  SPEAKER_01_cand2_start12.40.wav  (93.8 KB)
  SPEAKER_01_cand3_start15.76.wav  (77.6 KB)
  SPEAKER_01_cand4_start18.24.wav  (75.1 KB)
  SPEAKER_01_cand5_start20.64.wav  (67.6 KB)
  SPEAKER_01_

---

### Task 2: Voice Resolution Function

Write a pure function that resolves a speaker reference WAV path given a target language and optional speaker ID. This function will be used by the API to determine which voice file to pass to Chatterbox.

**File to create:** `foreign_whispers/voice_resolution.py`

**Resolution order:**
1. If speaker-specific WAV exists: `speakers/{lang}/{speaker_id}.wav`
2. If language default exists: `speakers/{lang}/default.wav`
3. Fall back to global: `speakers/default.wav`

#### 2.1 — Write the tests (TDD)

The tests below define the contract for `resolve_speaker_wav`. The implementation should pass all 5 cases: speaker-specific voice, language default, global default, language-only fallback, and unknown-language fallback.

In [6]:
# These tests define the contract for resolve_speaker_wav.
# DO NOT modify the tests — make your implementation pass them.

import tempfile
import os

def run_voice_tests():
    try:
        from foreign_whispers.voice_resolution import resolve_speaker_wav
    except (ImportError, ModuleNotFoundError):
        print("✗ foreign_whispers.voice_resolution not found — create the file first (Task 2.2)")
        return False

    passed, failed = 0, 0

    # Create a temporary speakers directory structure for testing
    with tempfile.TemporaryDirectory() as tmpdir:
        speakers = Path(tmpdir)

        # Create test structure:
        #   speakers/default.wav
        #   speakers/es/default.wav
        #   speakers/es/SPEAKER_00.wav
        #   speakers/fr/  (empty — no WAVs)
        (speakers / "default.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "es").mkdir()
        (speakers / "es" / "default.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "es" / "SPEAKER_00.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "fr").mkdir()

        # Test 1: Speaker-specific WAV exists
        try:
            result = resolve_speaker_wav(speakers, "es", "SPEAKER_00")
            assert result == "es/SPEAKER_00.wav", f"Expected 'es/SPEAKER_00.wav', got '{result}'"
            print("✓ Test 1 passed: speaker-specific WAV")
            passed += 1
        except Exception as e:
            print(f"✗ Test 1 FAILED: {e}")
            failed += 1

        # Test 2: No speaker-specific WAV, falls back to language default
        try:
            result = resolve_speaker_wav(speakers, "es", "SPEAKER_01")
            assert result == "es/default.wav", f"Expected 'es/default.wav', got '{result}'"
            print("✓ Test 2 passed: language default fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 2 FAILED: {e}")
            failed += 1

        # Test 3: No language dir WAVs, falls back to global default
        try:
            result = resolve_speaker_wav(speakers, "fr", "SPEAKER_00")
            assert result == "default.wav", f"Expected 'default.wav', got '{result}'"
            print("✓ Test 3 passed: global default fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 3 FAILED: {e}")
            failed += 1

        # Test 4: No speaker_id provided, uses language default
        try:
            result = resolve_speaker_wav(speakers, "es")
            assert result == "es/default.wav", f"Expected 'es/default.wav', got '{result}'"
            print("✓ Test 4 passed: no speaker_id uses language default")
            passed += 1
        except Exception as e:
            print(f"✗ Test 4 FAILED: {e}")
            failed += 1

        # Test 5: Unknown language, falls back to global default
        try:
            result = resolve_speaker_wav(speakers, "xx")
            assert result == "default.wav", f"Expected 'default.wav', got '{result}'"
            print("✓ Test 5 passed: unknown language fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 5 FAILED: {e}")
            failed += 1

    print(f"\n{'='*40}")
    print(f"Results: {passed} passed, {failed} failed")
    return failed == 0

# Run the voice resolution tests.
run_voice_tests()

✓ Test 1 passed: speaker-specific WAV
✓ Test 2 passed: language default fallback
✓ Test 3 passed: global default fallback
✓ Test 4 passed: no speaker_id uses language default
✓ Test 5 passed: unknown language fallback

Results: 5 passed, 0 failed


True

#### 2.2 — Implement `resolve_speaker_wav`

Create `foreign_whispers/voice_resolution.py` with the stub below. Replace the `raise NotImplementedError` with your implementation.

**Hints:**
1. The return value is a **relative path** (e.g. `"es/SPEAKER_00.wav"`) — this is what the Chatterbox container expects relative to `/app/voices/`
2. Check paths in order: speaker-specific → language default → global default
3. Use `Path.exists()` to test each candidate

#### 2.3 — Re-run the tests

After implementing `resolve_speaker_wav`, re-run the tests. All 5 should pass.

In [7]:
# Reload and re-run (only works after you've implemented the function)
try:
    import importlib
    import foreign_whispers.voice_resolution
    importlib.reload(foreign_whispers.voice_resolution)
    if run_voice_tests():
        print("\nAll tests passed!")
    else:
        print("\nSome tests failed — fix your implementation before continuing.")
except (ImportError, ModuleNotFoundError):
    print("Skipped — create foreign_whispers/voice_resolution.py first (Task 2.2)")

✓ Test 1 passed: speaker-specific WAV
✓ Test 2 passed: language default fallback
✓ Test 3 passed: global default fallback
✓ Test 4 passed: no speaker_id uses language default
✓ Test 5 passed: unknown language fallback

Results: 5 passed, 0 failed

All tests passed!


#### 2.4 — Commit

```bash
git add foreign_whispers/voice_resolution.py
git commit -m "feat: add resolve_speaker_wav voice resolution function"
```

---

### Task 3: Add `speaker_wav` Parameter to the TTS API

**Goal:** Expose speaker selection through the API so callers can choose a reference voice.

**Files to modify:**
- `api/src/core/config.py` — add `speakers_dir` property
- `api/src/routers/tts.py` — add `speaker_wav` query parameter
- `api/src/services/tts_service.py` and `api/src/services/tts_engine.py` — pass `speaker_wav` through to the Chatterbox client

#### 3.1 — Add `speakers_dir` to Settings

Open `api/src/core/config.py` and add this property:

```python
@property
def speakers_dir(self) -> Path:
    return self.data_dir.parent / "speakers"
```

#### 3.2 — Add `speaker_wav` to the TTS endpoint

Open `api/src/routers/tts.py`. Add a new query parameter and pass it through:

```python
from foreign_whispers.voice_resolution import resolve_speaker_wav

@router.post("/tts/{video_id}")
async def tts_endpoint(
    video_id: str,
    request: Request,
    config: str = Query(..., pattern=r"^c-[0-9a-f]{7}$"),
    alignment: bool = Query(False),
    speaker_wav: str = Query(None, description="Reference voice WAV path (e.g. 'es/default.wav')"),
):
```

Inside the endpoint, if `speaker_wav` is not provided, resolve it automatically:

```python
if speaker_wav is None:
    speaker_wav = resolve_speaker_wav(settings.speakers_dir, "es")
```

#### 3.3 — Pass `speaker_wav` through the service

Open `api/src/services/tts_service.py`. Modify `text_file_to_speech` to accept and forward `speaker_wav`:

```python
def text_file_to_speech(self, source_path: str, output_path: str,
                        *, alignment: bool | None = None,
                        speaker_wav: str | None = None) -> None:
    tts_text_file_to_speech(source_path, output_path, self.tts_engine,
                            alignment=alignment, speaker_wav=speaker_wav)
```

Then in `api/src/services/tts_engine.py`, the `text_file_to_speech` function must pass `speaker_wav` as a kwarg to `ChatterboxClient.tts_to_file()`. Find where `tts_to_file` is called and add:

```python
client.tts_to_file(text, wav_path, speaker_wav=speaker_wav)
```

#### 3.4 — Test manually

I did not rebuild or restart from this notebook. The API integration was validated with static wiring checks and regression tests.

In [8]:
# Task 3: Verify speaker_wav is wired through API/router/service layers

files_to_check = [
    PROJECT_ROOT / "api" / "src" / "core" / "config.py",
    PROJECT_ROOT / "api" / "src" / "routers" / "tts.py",
    PROJECT_ROOT / "api" / "src" / "services" / "tts_service.py",
    PROJECT_ROOT / "api" / "src" / "services" / "tts_engine.py",
]

keywords = [
    "speakers_dir",
    "speaker_wav",
    "speaker_mapping",
    "resolve_speaker_wav",
]

for path in files_to_check:
    print("\n" + "=" * 100)
    print(path.relative_to(PROJECT_ROOT))
    print("=" * 100)

    lines = path.read_text().splitlines()
    for i, line in enumerate(lines, start=1):
        if any(k in line for k in keywords):
            print(f"{i:4d}: {line}")


api/src/core/config.py
  59:     def speakers_dir(self) -> Path:

api/src/routers/tts.py
  31:     speaker_wav: str | None = Query(None),
  63:     speaker_mapping = {}
  66:     if speaker_wav:
  67:         # Explicit speaker_wav query parameter overrides automatic per-speaker mapping.
  78:             speaker_mapping[spk] = speaker_wav
  87:         speakers_dir = settings.speakers_dir / lang
  89:         if speakers_dir.exists():
  90:             available_voices = sorted(list(speakers_dir.glob("*.wav")))
 103:                         speaker_mapping[spk] = str(available_voices[0])
 115:                         speaker_mapping[spk] = str(assigned_voice)
 120:         None, svc.text_file_to_speech, source_path, str(audio_dir), alignment=alignment, speaker_mapping=speaker_mapping

api/src/services/tts_service.py
  26:         speaker_mapping: dict[str, str] | None = None
  29:         # Keep backward compatibility with callers/tests that do not accept speaker_mapping.
  32:      

Task 3 result:

The API layer now exposes speaker selection and passes it through to the TTS service. The router derives the speaker reference directory from `settings.data_dir.parent / "speakers"`. The TTS router can accept speaker-specific voice information and the service forwards `speaker_mapping` to the TTS engine. The engine eventually calls `ChatterboxClient.tts_to_file(..., speaker_wav=...)`, which triggers the `/v1/audio/speech/upload` path.

#### 3.5 — Commit

```bash
git add api/src/core/config.py api/src/routers/tts.py api/src/services/tts_service.py api/src/services/tts_engine.py
git commit -m "feat: add speaker_wav parameter to TTS API endpoint"
```

---

### Task 4: Per-Speaker Voice Assignment

**Goal:** When transcription segments have `speaker` labels (from diarization), automatically assign different reference voices to different speakers.

**Prerequisite:** Task 5 from the `diarization_integration` notebook (segments have `speaker` field).

**File to modify:** `api/src/routers/tts.py`

#### 4.1 — Build a speaker-to-voice mapping

In the TTS endpoint, before calling TTS, read the translated transcript and extract unique speakers. Map each speaker to a reference WAV using `resolve_speaker_wav`:

```python
import json

# Load translated transcript to get speaker labels
trans_path = settings.translations_dir / f"{title}.json"
translated = json.loads(trans_path.read_text())
segments = translated.get("segments", [])

# Build speaker → voice mapping
unique_speakers = sorted(set(seg.get("speaker", "SPEAKER_00") for seg in segments))
voice_map = {
    spk: resolve_speaker_wav(settings.speakers_dir, "es", spk)
    for spk in unique_speakers
}
```

#### 4.2 — Modify `api/src/services/tts_engine.py` to accept per-segment speaker hints

Currently `text_file_to_speech` synthesizes all segments with one `speaker_wav`. You need to modify it so that each segment can use a different speaker_wav based on its `speaker` field.

**Approach:** Pass `voice_map` as a dict to `text_file_to_speech`. Inside the function, for each segment, look up `voice_map[segment["speaker"]]` and pass it as `speaker_wav` to `tts_to_file()`.

This is the most open-ended part of the assignment. Study how `text_file_to_speech` iterates over segments in `api/src/services/tts_engine.py` and decide where to inject the per-segment voice selection.

#### 4.3 — Test with a multi-speaker video

After implementing per-speaker voice assignment:

1. Run the diarization integration first to label segments with speaker IDs
2. Then run TTS — each speaker should use a different reference voice
3. Listen to the output to verify voice switching works

In [9]:
# Task 4: Verify per-speaker voice assignment on the final demo transcript

import json
from pathlib import Path

TITLE = "Strait of Hormuz disruption threatens to shake global economy"
LANGUAGE = "es"

trans_path = (
    PROJECT_ROOT
    / "pipeline_data"
    / "api"
    / "translations"
    / "argos"
    / f"{TITLE}.json"
)

print("Translation path:", trans_path)
print("exists:", trans_path.exists())

translated = json.loads(trans_path.read_text())
segments = translated.get("segments", [])

speakers = sorted({seg.get("speaker") for seg in segments if seg.get("speaker")})
labeled = sum(1 for seg in segments if seg.get("speaker"))

print(f"Total segments: {len(segments)}")
print(f"With speaker labels: {labeled}")
print(f"Unique speakers: {speakers}")

speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"
lang_dir = speakers_dir / LANGUAGE

print("\nAvailable reference WAVs:")
for wav in sorted(lang_dir.glob("*.wav")):
    print(f"  {wav.name} ({wav.stat().st_size / 1024:.1f} KB)")

# This is the mapping used for the final HPC-generated demo.
# The samples were manually validated from short clips before synthesis.
manual_mapping = {
    "SPEAKER_00": "es/voice_0.wav",
    "SPEAKER_01": "es/voice_1.wav",
    "SPEAKER_02": "es/voice_2.wav",
}

print("\nFinal demo speaker-to-voice mapping:")
for spk in speakers:
    print(f"  {spk} -> {manual_mapping.get(spk, 'fallback/default voice')}")

Translation path: /root/foreign-whispers/pipeline_data/api/translations/argos/Strait of Hormuz disruption threatens to shake global economy.json
exists: True
Total segments: 170
With speaker labels: 170
Unique speakers: ['SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02']

Available reference WAVs:
  voice_0.wav (93.8 KB)
  voice_1.wav (93.8 KB)
  voice_2.wav (93.8 KB)

Final demo speaker-to-voice mapping:
  SPEAKER_00 -> es/voice_0.wav
  SPEAKER_01 -> es/voice_1.wav
  SPEAKER_02 -> es/voice_2.wav


Task 4 result:

The final transcript contains speaker labels for all 170 translated segments. For the final demo, I manually validated the diarized speaker identities and selected clean reference samples:
- SPEAKER_00 → voice_0.wav, female host voice
- SPEAKER_01 → voice_1.wav, male interviewee 1
- SPEAKER_02 → voice_2.wav, male interviewee 2

The HPC-generated Chatterbox run used this mapping to synthesize per-speaker Spanish TTS.

#### 4.4 — Commit

```bash
git add api/src/routers/tts.py api/src/services/tts_service.py api/src/services/tts_engine.py
git commit -m "feat: per-speaker voice assignment in TTS"
```

---

## Evaluation Criteria

| # | Criterion | How to verify |
| - | --------- | ------------- |
| 1 | Tests pass | Re-run voice resolution tests — all 5 green |
| 2 | API accepts `speaker_wav` | `POST /api/tts/{video_id}?speaker_wav=es/default.wav` works |
| 3 | Auto-resolution works | Omitting `speaker_wav` selects language default automatically |
| 4 | Per-speaker mapping | With diarized segments, different speakers get different voices |
| 5 | Fallback chain | Unknown speaker/language falls back to `default.wav` |
| 6 | Code quality | Follows existing patterns (query params, service layer, config properties) |

## Notebook 6 Summary

This notebook validates the TTS integration for the final demo clip. I did not re-run full Chatterbox synthesis inside the notebook because the final audio was generated on the HPC GPU and copied back into the project. Instead, I verified the generated WAV and final remuxed MP4 artifacts.

The voice resolution function passed all 5 TDD tests. The API and service layers are wired to pass speaker-specific voice information down to the Chatterbox client, which supports reference WAV uploads through `/v1/audio/speech/upload`.

For the final demo, all 170 translated segments have speaker labels. The final speaker-to-voice mapping is:
- SPEAKER_00 → `es/voice_0.wav`
- SPEAKER_01 → `es/voice_1.wav`
- SPEAKER_02 → `es/voice_2.wav`

The final video plays correctly with per-speaker voice cloning and aligned Spanish TTS.